# Phase 2 - Session-Level Behavioural Feature Engineering and Classification

To classify smartphone-use behavior into interpretable behavioural states using literature grounded rules.

## File Imports

In [ ]:
import pandas as pd
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Read CSV file
try:
  df = pd.read_excel("/content/drive/My Drive/Research Thesis/lsapp_final.xlsx")
  print("File read successfully")
except FileNotFoundError:
  print("File not found")

File read successfully


In [ ]:
df

,user_id,session_id,episode_number,app_name,timestamp,event_class,start_time,end_time,duration_seconds,session_event_number
0,0,1,1.0,Minesweeper Classic (Mines),2018-01-16 06:01:05,episode,2018-01-16 06:01:05,2018-01-16 06:01:05,0.0,1
1,0,1,2.0,Minesweeper Classic (Mines),2018-01-16 06:01:07,episode,2018-01-16 06:01:07,2018-01-16 06:01:07,0.0,2
2,0,1,3.0,Minesweeper Classic (Mines),2018-01-16 06:01:08,episode,2018-01-16 06:01:08,2018-01-16 06:01:08,0.0,3
3,0,1,4.0,Minesweeper Classic (Mines),2018-01-16 06:01:09,episode,2018-01-16 06:01:09,2018-01-16 06:01:09,0.0,4
4,0,1,5.0,Minesweeper Classic (Mines),2018-01-16 06:03:44,episode,2018-01-16 06:03:44,2018-01-16 06:03:44,0.0,5
...,...,...,...,...,...,...,...,...,...,...
899344,291,76246,1.0,Settings,2018-04-06 13:20:12,episode,2018-04-06 13:20:12,2018-04-06 13:20:14,2.0,1
899345,291,76247,1.0,Settings,2018-04-06 14:34:15,episode,2018-04-06 14:34:15,2018-04-06 14:34:17,2.0,1
899346,291,76247,2.0,Facebook,2018-04-06 14:34:34,episode,2018-04-06 14:34:34,2018-04-06 14:35:15,41.0,2
899347,291,76247,3.0,Facebook,2018-04-06 14:35:15,episode,2018-04-06 14:35:15,2018-04-06 14:35:37,22.0,3


## Dataset Audit

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899349 entries, 0 to 899348
Data columns (total 10 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   user_id               899349 non-null  int64         
 1   session_id            899349 non-null  int64         
 2   episode_number        585520 non-null  float64       
 3   app_name              899349 non-null  object        
 4   timestamp             899349 non-null  datetime64[ns]
 5   event_class           899349 non-null  object        
 6   start_time            585520 non-null  datetime64[ns]
 7   end_time              585520 non-null  datetime64[ns]
 8   duration_seconds      585520 non-null  float64       
 9   session_event_number  899349 non-null  int64         
dtypes: datetime64[ns](3), float64(2), int64(3), object(2)
memory usage: 68.6+ MB


In [ ]:
# Count the records belonging to each event class
df["event_class"].value_counts(dropna=False)

,count
event_class,
episode,585520
interaction,313829


In [ ]:
# Check whether duration is available for each event class
pd.crosstab(
    df["event_class"],
    df["duration_seconds"].notna(),
    rownames=["Event Class"],
    colnames=["Duration Available"]
)

Duration Available,False,True
Event Class,,
episode,0,585520
interaction,313829,0


In [ ]:

# Count unique users
n_users = df["user_id"].nunique()

# Count distinct sessions using both user and session IDs
n_sessions = df[
    ["user_id", "session_id"]
].drop_duplicates().shape[0]

print("Unique users:", n_users)
print("Unique user-session combinations:", n_sessions)

Unique users: 291
Unique user-session combinations: 70979


In [ ]:

# Select episode records
episodes = df[df["event_class"] == "episode"]

# Identify episodes with zero duration
zero_episodes = episodes[
    episodes["duration_seconds"] == 0
]

# Calculate their prevalence
print("Total episodes:", len(episodes))
print("Zero-duration episodes:", len(zero_episodes))

print(
    "Percentage:",
    round(len(zero_episodes) / len(episodes) * 100, 2),
    "%"
)

# Inspect a sample of zero-duration episodes
display(
    zero_episodes[
        ["user_id", "session_id", "app_name",
         "start_time", "end_time", "duration_seconds"]
    ].head(10)
)

Total episodes: 585520
Zero-duration episodes: 55073
Percentage: 9.41 %


,user_id,session_id,app_name,start_time,end_time,duration_seconds
0,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:05,2018-01-16 06:01:05,0.0
1,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:07,2018-01-16 06:01:07,0.0
2,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:08,2018-01-16 06:01:08,0.0
3,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:09,2018-01-16 06:01:09,0.0
4,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:44,2018-01-16 06:03:44,0.0
5,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:45,2018-01-16 06:03:45,0.0
6,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:47,2018-01-16 06:03:47,0.0
7,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:49,2018-01-16 06:03:49,0.0
8,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:51,2018-01-16 06:03:51,0.0
9,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:52,2018-01-16 06:03:52,0.0


In [ ]:

# Check whether zero-duration episodes have identical
# start and end timestamps

same_timestamp = (
    zero_episodes["start_time"] ==
    zero_episodes["end_time"]
)

print("Zero-duration episodes:", len(zero_episodes))
print("Identical start/end timestamps:", same_timestamp.sum())
print("Different start/end timestamps:", (~same_timestamp).sum())

# Inspect 10 examples
display(
    zero_episodes[
        ["user_id", "session_id", "app_name",
         "start_time", "end_time", "duration_seconds"]
    ].head(10)
)

Zero-duration episodes: 55073
Identical start/end timestamps: 55073
Different start/end timestamps: 0


,user_id,session_id,app_name,start_time,end_time,duration_seconds
0,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:05,2018-01-16 06:01:05,0.0
1,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:07,2018-01-16 06:01:07,0.0
2,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:08,2018-01-16 06:01:08,0.0
3,0,1,Minesweeper Classic (Mines),2018-01-16 06:01:09,2018-01-16 06:01:09,0.0
4,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:44,2018-01-16 06:03:44,0.0
5,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:45,2018-01-16 06:03:45,0.0
6,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:47,2018-01-16 06:03:47,0.0
7,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:49,2018-01-16 06:03:49,0.0
8,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:51,2018-01-16 06:03:51,0.0
9,0,1,Minesweeper Classic (Mines),2018-01-16 06:03:52,2018-01-16 06:03:52,0.0


In [ ]:

# Define our session identifiers
session_keys = ["user_id", "session_id"]

# Identify episodes with positive durations
valid_episodes = episodes[
    episodes["duration_seconds"] > 0
]

# Count distinct sessions in the original dataset
total_sessions = df[session_keys].drop_duplicates().shape[0]

# Count sessions containing zero-duration episodes
affected_sessions = zero_episodes[
    session_keys
].drop_duplicates().shape[0]

# Count sessions containing positive-duration episodes
valid_sessions = valid_episodes[
    session_keys
].drop_duplicates().shape[0]

print("Total sessions:", total_sessions)
print("Sessions with zero-duration episodes:", affected_sessions)
print("Sessions with positive-duration episodes:", valid_sessions)
print(
    "Sessions without positive-duration episodes:",
    total_sessions - valid_sessions
)

# Investigate applications most affected
display(
    zero_episodes["app_name"]
    .value_counts()
    .head(10)
)

Total sessions: 70979
Sessions with zero-duration episodes: 23506
Sessions with positive-duration episodes: 55793
Sessions without positive-duration episodes: 15186


,count
app_name,
Google Chrome,11776
Facebook Messenger,5075
Minesweeper Classic (Mines),2873
Slidejoy,2752
Instagram,2255
Samsung Pay,2250
DigiHUD Pro Speedometer,2017
Facebook,1844
Google,1752


In [ ]:

# Identify the types of records present in each session
session_types = (
    df.assign(
        positive_episode=(
            df["event_class"].eq("episode") &
            df["duration_seconds"].gt(0)
        ),
        zero_episode=(
            df["event_class"].eq("episode") &
            df["duration_seconds"].eq(0)
        ),
        interaction=df["event_class"].eq("interaction")
    )
    .groupby(["user_id", "session_id"])[
        ["positive_episode", "zero_episode", "interaction"]
    ]
    .any()
)

# Focus on sessions without positive-duration episodes
no_positive = session_types[
    ~session_types["positive_episode"]
]

print(
    "Only zero-duration episodes:",
    (no_positive["zero_episode"] &
     ~no_positive["interaction"]).sum()
)

print(
    "Zero-duration episodes and interactions:",
    (no_positive["zero_episode"] &
     no_positive["interaction"]).sum()
)

print(
    "Only interactions:",
    (~no_positive["zero_episode"] &
     no_positive["interaction"]).sum()
)

Only zero-duration episodes: 1919
Zero-duration episodes and interactions: 985
Only interactions: 12282


In [ ]:

# Count timestamps containing fractional seconds
fractional_timestamps = (
    df["timestamp"].dt.microsecond != 0
).sum()

print("Timestamps with milliseconds:", fractional_timestamps)

Timestamps with milliseconds: 0


In [ ]:

# Select sessions containing only interactions
interaction_only = session_types[
    ~session_types["positive_episode"] &
    ~session_types["zero_episode"] &
    session_types["interaction"]
]

# Calculate earliest and latest timestamps for each session
session_spans = (
    df.groupby(["user_id", "session_id"])["timestamp"]
    .agg(["min", "max"])
)

# Calculate elapsed time between recorded events
session_spans["span_seconds"] = (
    session_spans["max"] - session_spans["min"]
).dt.total_seconds()

# Examine interaction-only sessions
interaction_spans = session_spans.loc[interaction_only.index]

print(
    "Interaction-only sessions:",
    len(interaction_spans)
)

print(
    "Sessions with positive event span:",
    (interaction_spans["span_seconds"] > 0).sum()
)

print(
    "Sessions with zero event span:",
    (interaction_spans["span_seconds"] == 0).sum()
)

Interaction-only sessions: 12282
Sessions with positive event span: 4370
Sessions with zero event span: 7912



### Data Quality Decisions

Based on the preliminary dataset audit:

1. Zero-duration episodes will be retained because
   their recorded durations may be affected by
   second-level timestamp precision.

2. Interaction-only sessions will be preserved.

3. Episode duration and session event span will
   be calculated separately.

4. Parry and Toth's glances cannot be reliably
   reconstructed from the available LSApp fields.

5. Short app-use frequency will be distinguished
   from overall session frequency.

6. The original dataset will remain unchanged.

## Constructing the basic session-level DataFrame

### Session aggregration

In [ ]:
# Create indicators for the different event types
events = df.assign(
    is_episode=df["event_class"].eq("episode").astype(int),
    is_interaction=df["event_class"].eq("interaction").astype(int),
    is_zero_episode=(
        df["event_class"].eq("episode") &
        df["duration_seconds"].eq(0)
    ).astype(int)
)

# Aggregate by user and session
sessions = (
    events.groupby(
        ["user_id", "session_id"],
        as_index=False
    )
    .agg(
        first_event_time=("timestamp", "min"),
        last_event_time=("timestamp", "max"),
        event_count=("event_class", "size"),
        episode_count=("is_episode", "sum"),
        interaction_count=("is_interaction", "sum"),
        zero_duration_episode_count=("is_zero_episode", "sum")
    )
)

# Calculate the observed event span
sessions["event_span_seconds"] = (
    sessions["last_event_time"] -
    sessions["first_event_time"]
).dt.total_seconds()

print("Aggregated dataset shape:", sessions.shape)

display(sessions.head(10))

Aggregated dataset shape: (70979, 9)


,user_id,session_id,first_event_time,last_event_time,event_count,episode_count,interaction_count,zero_duration_episode_count,event_span_seconds
0,0,1,2018-01-16 06:01:05,2018-01-16 06:04:17,31,31,0,31,192.0
1,0,2,2018-01-16 06:25:54,2018-01-16 06:26:21,4,3,1,0,27.0
2,0,3,2018-01-16 06:35:35,2018-01-16 06:35:35,1,1,0,0,0.0
3,0,4,2018-01-16 07:15:56,2018-01-16 07:21:44,36,36,0,35,348.0
4,0,5,2018-01-16 08:02:05,2018-01-16 08:04:11,39,39,0,39,126.0
5,0,6,2018-01-16 08:30:44,2018-01-16 08:34:14,41,41,0,40,210.0
6,0,7,2018-01-16 09:45:51,2018-01-16 09:51:45,40,40,0,40,354.0
7,0,8,2018-01-16 09:59:57,2018-01-16 09:59:57,2,0,2,0,0.0
8,0,9,2018-01-16 10:29:54,2018-01-16 10:29:54,1,0,1,0,0.0
9,0,10,2018-01-16 11:01:28,2018-01-16 11:03:27,32,32,0,31,119.0


In [ ]:

# Validate aggregated record counts
print("Original records:", len(df))
print("Aggregated sessions:", len(sessions))

print(
    "Total events accounted for:",
    sessions["event_count"].sum()
)

print(
    "Total episodes:",
    sessions["episode_count"].sum()
)

print(
    "Total interactions:",
    sessions["interaction_count"].sum()
)

# Verify data integrity
assert len(sessions) == n_sessions
assert sessions["event_count"].sum() == len(df)
assert (
    sessions["episode_count"] +
    sessions["interaction_count"] ==
    sessions["event_count"]
).all()

print("All aggregation checks passed!")

Original records: 899349
Aggregated sessions: 70979
Total events accounted for: 899349
Total episodes: 585520
Total interactions: 313829
All aggregation checks passed!


### Recorded Episode Duration

In [ ]:
# Group episode records and sum their durations
episode_durations = (
    df[df["event_class"] == "episode"]
    .groupby(["user_id", "session_id"], as_index=False)
    .agg(
        recorded_episode_seconds=("duration_seconds", "sum")
    )
)

# Add episode duration to our session-level dataset
sessions = sessions.merge(
    episode_durations,
    on=["user_id", "session_id"],
    how="left",
    validate="one_to_one"
)

# Display the result
display(
    sessions[
        ["user_id", "session_id",
         "event_span_seconds", "recorded_episode_seconds"]
    ].head(10)
)

,user_id,session_id,event_span_seconds,recorded_episode_seconds
0,0,1,192.0,0.0
1,0,2,27.0,21.0
2,0,3,0.0,51.0
3,0,4,348.0,1.0
4,0,5,126.0,0.0
5,0,6,210.0,1.0
6,0,7,354.0,0.0
7,0,8,0.0,NaN
8,0,9,0.0,NaN
9,0,10,119.0,1.0


In [ ]:

# Check the number of sessions
print("Total sessions:", len(sessions))

# Count sessions without recorded episode duration
print(
    "Sessions with missing episode duration:",
    sessions["recorded_episode_seconds"].isna().sum()
)

# Count sessions with exactly zero recorded duration
print(
    "Sessions with zero recorded episode duration:",
    sessions["recorded_episode_seconds"].eq(0).sum()
)

# Confirm the total episode duration was preserved
print(
    "Original total episode seconds:",
    df["duration_seconds"].sum()
)

print(
    "Aggregated total episode seconds:",
    sessions["recorded_episode_seconds"].sum()
)

assert len(sessions) == n_sessions

assert (
    sessions["recorded_episode_seconds"].sum()
    == df["duration_seconds"].sum()
)

print("Duration aggregation validated!")

Total sessions: 70979
Sessions with missing episode duration: 12282
Sessions with zero recorded episode duration: 2904
Original total episode seconds: 14161098.0
Aggregated total episode seconds: 14161098.0
Duration aggregation validated!


In [ ]:

# Determine the observed start and end of each record
df["observed_start"] = df["start_time"].fillna(df["timestamp"])
df["observed_end"] = df["end_time"].fillna(df["timestamp"])

# Find the earliest start and latest end per session
session_boundaries = (
    df.groupby(["user_id", "session_id"], as_index=False)
    .agg(
        session_start=("observed_start", "min"),
        session_end=("observed_end", "max")
    )
)

# Add boundaries to the session-level dataset
sessions = sessions.merge(
    session_boundaries,
    on=["user_id", "session_id"],
    how="left",
    validate="one_to_one"
)

# Calculate the full observed session span
sessions["session_span_seconds"] = (
    sessions["session_end"] -
    sessions["session_start"]
).dt.total_seconds()

# Compare the three duration measurements
display(
    sessions[
        ["user_id", "session_id",
         "event_span_seconds",
         "recorded_episode_seconds",
         "session_span_seconds"]
    ].head(10)
)

,user_id,session_id,event_span_seconds,recorded_episode_seconds,session_span_seconds
0,0,1,192.0,0.0,192.0
1,0,2,27.0,21.0,32.0
2,0,3,0.0,51.0,51.0
3,0,4,348.0,1.0,348.0
4,0,5,126.0,0.0,126.0
5,0,6,210.0,1.0,210.0
6,0,7,354.0,0.0,354.0
7,0,8,0.0,NaN,0.0
8,0,9,0.0,NaN,0.0
9,0,10,119.0,1.0,119.0


In [ ]:
# Validate session boundaries and spans
assert len(sessions) == n_sessions

assert (
    sessions["session_span_seconds"] >= 0
).all()

assert (
    sessions["session_span_seconds"] >=
    sessions["event_span_seconds"]
).all()

print("Session span validation passed!")

Session span validation passed!


### Application-level behavioural feature engineering

In [ ]:

# Define the grouping identifiers
session_keys = ["user_id", "session_id"]

# Count unique applications across all event records
app_diversity = (
    df.groupby(session_keys, as_index=False)
    .agg(
        unique_apps_observed=("app_name", "nunique")
    )
)

# Count unique applications from episode records only
episode_app_diversity = (
    df[df["event_class"] == "episode"]
    .groupby(session_keys, as_index=False)
    .agg(
        unique_episode_apps=("app_name", "nunique")
    )
)

# Merge the features into our session-level dataset
sessions = sessions.merge(
    app_diversity,
    on=session_keys,
    how="left",
    validate="one_to_one"
)

sessions = sessions.merge(
    episode_app_diversity,
    on=session_keys,
    how="left",
    validate="one_to_one"
)

# Sessions without episodes have zero episode-based apps
sessions["unique_episode_apps"] = (
    sessions["unique_episode_apps"]
    .fillna(0)
    .astype(int)
)

display(
    sessions[
        ["user_id", "session_id", "episode_count",
         "interaction_count", "unique_apps_observed",
         "unique_episode_apps"]
    ].head(10)
)

,user_id,session_id,episode_count,interaction_count,unique_apps_observed,unique_episode_apps
0,0,1,31,0,1,1
1,0,2,3,1,3,2
2,0,3,1,0,1,1
3,0,4,36,0,2,2
4,0,5,39,0,2,2
5,0,6,41,0,2,2
6,0,7,40,0,2,2
7,0,8,0,2,1,0
8,0,9,0,1,1,0
9,0,10,32,0,2,2


In [ ]:

# Confirm no sessions were lost
assert len(sessions) == n_sessions

# Unique applications cannot exceed recorded event count
assert (
    sessions["unique_apps_observed"] <=
    sessions["event_count"]
).all()

# Episode app diversity cannot exceed episode count
assert (
    sessions["unique_episode_apps"] <=
    sessions["episode_count"]
).all()

# Apps observed across all events must include episode apps
assert (
    sessions["unique_apps_observed"] >=
    sessions["unique_episode_apps"]
).all()

print("Application diversity validation passed!")

# Summary statistics
display(
    sessions[
        ["unique_apps_observed", "unique_episode_apps"]
    ].describe()
)

# Count sessions where the measurements differ
different = (
    sessions["unique_apps_observed"] !=
    sessions["unique_episode_apps"]
).sum()

print("Sessions with different app diversity counts:", different)

Application diversity validation passed!


,unique_apps_observed,unique_episode_apps
count,70979.000000,70979.000000
mean,2.395455,1.815396
std,1.676191,1.575073
min,1.000000,0.000000
25%,1.000000,1.000000
50%,2.000000,1.000000
75%,3.000000,3.000000
max,17.000000,16.000000


Sessions with different app diversity counts: 31812


In [ ]:
# Count distinct apps appearing in interaction records
interaction_app_diversity = (
    df[df["event_class"] == "interaction"]
    .groupby(session_keys, as_index=False)
    .agg(
        unique_interaction_apps=("app_name", "nunique")
    )
)

# Merge into the session-level dataset
sessions = sessions.merge(
    interaction_app_diversity,
    on=session_keys,
    how="left",
    validate="one_to_one"
)

# Sessions without interactions have zero interaction apps
sessions["unique_interaction_apps"] = (
    sessions["unique_interaction_apps"]
    .fillna(0)
    .astype(int)
)

# Inspect the three diversity measures
display(
    sessions[
        ["unique_apps_observed",
         "unique_episode_apps",
         "unique_interaction_apps"]
    ].describe()
)

,unique_apps_observed,unique_episode_apps,unique_interaction_apps
count,70979.000000,70979.000000,70979.000000
mean,2.395455,1.815396,0.972288
std,1.676191,1.575073,1.083073
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,0.000000
50%,2.000000,1.000000,1.000000
75%,3.000000,3.000000,1.000000
max,17.000000,16.000000,12.000000


### App Switches and Revists

In [ ]:
# Define session identifiers
session_keys = ["user_id", "session_id"]

# Arrange all events by their recorded order
ordered_events = (
    df.sort_values(
        session_keys + ["session_event_number"],
        kind="stable"
    )
    .copy()
)

# Find the previously recorded app within each session
ordered_events["previous_app"] = (
    ordered_events
    .groupby(session_keys)["app_name"]
    .shift(1)
)

# Identify changes between consecutive app names
ordered_events["is_app_transition"] = (
    ordered_events["previous_app"].notna()
    & ordered_events["app_name"].notna()
    & (
        ordered_events["app_name"]
        != ordered_events["previous_app"]
    )
)

# Identify returns to previously observed apps
ordered_events["is_app_revisit"] = (
    ordered_events["is_app_transition"]
    & ordered_events.duplicated(
        subset=session_keys + ["app_name"]
    )
)

# Inspect the results
display(
    ordered_events[
        ["user_id", "session_id",
         "session_event_number", "event_class",
         "app_name", "previous_app",
         "is_app_transition", "is_app_revisit"]
    ].tail(20)
)

,user_id,session_id,session_event_number,event_class,app_name,previous_app,is_app_transition,is_app_revisit
899329,291,76243,7,episode,Facebook Messenger,Facebook Messenger,False,False
899330,291,76243,8,episode,Facebook Messenger,Facebook Messenger,False,False
899331,291,76243,9,episode,Facebook Messenger,Facebook Messenger,False,False
899332,291,76243,10,interaction,Facebook Messenger,Facebook Messenger,False,False
899333,291,76243,11,interaction,Facebook Messenger,Facebook Messenger,False,False
899334,291,76243,12,interaction,Facebook Messenger,Facebook Messenger,False,False
899335,291,76243,13,episode,Facebook Messenger,Facebook Messenger,False,False
899336,291,76243,14,interaction,Facebook Messenger,Facebook Messenger,False,False
899337,291,76243,15,interaction,Facebook Messenger,Facebook Messenger,False,False
899338,291,76243,16,interaction,Facebook Messenger,Facebook Messenger,False,False


In [ ]:

# Identify duplicate event-order numbers within sessions
duplicate_orders = df.duplicated(
    subset=session_keys + ["session_event_number"]
).sum()

print("Duplicate session event-order identifiers:",
      duplicate_orders)

# Count event-level transitions and revisits
print(
    "Observed transitions:",
    ordered_events["is_app_transition"].sum()
)

print(
    "Observed revisits:",
    ordered_events["is_app_revisit"].sum()
)

# A revisit must also be a transition
assert (
    ~ordered_events["is_app_revisit"]
    | ordered_events["is_app_transition"]
).all()

print("Transition/revisit logic validated!")

Duplicate session event-order identifiers: 0
Observed transitions: 234479
Observed revisits: 135431
Transition/revisit logic validated!


In [ ]:

# Aggregate application navigation indicators by session
app_navigation = (
    ordered_events
    .groupby(["user_id", "session_id"], as_index=False)
    .agg(
        observed_app_transition_count=(
            "is_app_transition", "sum"
        ),
        observed_app_revisit_count=(
            "is_app_revisit", "sum"
        )
    )
)

# Merge navigation features into our session dataset
sessions = sessions.merge(
    app_navigation,
    on=["user_id", "session_id"],
    how="left",
    validate="one_to_one"
)

# Inspect the results
display(
    sessions[
        ["user_id", "session_id",
         "event_count",
         "unique_apps_observed",
         "observed_app_transition_count",
         "observed_app_revisit_count"]
    ].head(10)
)

,user_id,session_id,event_count,unique_apps_observed,observed_app_transition_count,observed_app_revisit_count
0,0,1,31,1,0,0
1,0,2,4,3,3,1
2,0,3,1,1,0,0
3,0,4,36,2,3,2
4,0,5,39,2,3,2
5,0,6,41,2,3,2
6,0,7,40,2,3,2
7,0,8,2,1,0,0
8,0,9,1,1,0,0
9,0,10,32,2,3,2


In [ ]:

# Validate session preservation
assert len(sessions) == n_sessions

# Verify transition totals
assert (
    sessions["observed_app_transition_count"].sum()
    == ordered_events["is_app_transition"].sum()
)

# Verify revisit totals
assert (
    sessions["observed_app_revisit_count"].sum()
    == ordered_events["is_app_revisit"].sum()
)

# A session cannot have more transitions than
# the number of gaps between consecutive events
assert (
    sessions["observed_app_transition_count"]
    <= sessions["event_count"] - 1
).all()

# Revisits must be a subset of transitions
assert (
    sessions["observed_app_revisit_count"]
    <= sessions["observed_app_transition_count"]
).all()

# Every session should have a recorded count
assert sessions[
    ["observed_app_transition_count",
     "observed_app_revisit_count"]
].notna().all().all()

print("Application navigation aggregation validated!")

# Examine the distribution across sessions
display(
    sessions[
        ["observed_app_transition_count",
         "observed_app_revisit_count"]
    ].describe()
)

Application navigation aggregation validated!


,observed_app_transition_count,observed_app_revisit_count
count,70979.000000,70979.000000
mean,3.303498,1.908043
std,6.803626,5.680851
min,0.000000,0.000000
25%,0.000000,0.000000
50%,1.000000,0.000000
75%,4.000000,2.000000
max,218.000000,209.000000


### Temporal behavioural features

In [ ]:
sessions.sort_values(
        ["user_id", "session_start", "session_id"]
    ).reset_index(drop=True)

,user_id,session_id,first_event_time,last_event_time,event_count,episode_count,interaction_count,zero_duration_episode_count,event_span_seconds,recorded_episode_seconds,session_start,session_end,session_span_seconds,unique_apps_observed,unique_episode_apps,unique_interaction_apps,observed_app_transition_count,observed_app_revisit_count
0,0,1,2018-01-16 06:01:05,2018-01-16 06:04:17,31,31,0,31,192.0,0.0,2018-01-16 06:01:05,2018-01-16 06:04:17,192.0,1,1,0,0,0
1,0,2,2018-01-16 06:25:54,2018-01-16 06:26:21,4,3,1,0,27.0,21.0,2018-01-16 06:25:54,2018-01-16 06:26:26,32.0,3,2,1,3,1
2,0,3,2018-01-16 06:35:35,2018-01-16 06:35:35,1,1,0,0,0.0,51.0,2018-01-16 06:35:35,2018-01-16 06:36:26,51.0,1,1,0,0,0
3,0,4,2018-01-16 07:15:56,2018-01-16 07:21:44,36,36,0,35,348.0,1.0,2018-01-16 07:15:56,2018-01-16 07:21:44,348.0,2,2,0,3,2
4,0,5,2018-01-16 08:02:05,2018-01-16 08:04:11,39,39,0,39,126.0,0.0,2018-01-16 08:02:05,2018-01-16 08:04:11,126.0,2,2,0,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70974,291,76243,2018-04-06 12:04:38,2018-04-06 12:24:15,18,10,8,1,1177.0,849.0,2018-04-06 12:04:38,2018-04-06 12:24:15,1177.0,3,2,2,4,2
70975,291,76244,2018-04-06 12:38:32,2018-04-06 12:38:32,1,0,1,0,0.0,NaN,2018-04-06 12:38:32,2018-04-06 12:38:32,0.0,1,0,1,0,0
70976,291,76245,2018-04-06 12:47:28,2018-04-06 12:49:30,2,1,1,0,122.0,345.0,2018-04-06 12:47:28,2018-04-06 12:53:13,345.0,1,1,1,0,0
70977,291,76246,2018-04-06 13:20:12,2018-04-06 13:20:12,1,1,0,0,0.0,2.0,2018-04-06 13:20:12,2018-04-06 13:20:14,2.0,1,1,0,0,0


In [ ]:
# Calculate inter-session gaps
# Sort sessions chronologically for each user
sessions = (
    sessions.sort_values(
        ["user_id", "session_start", "session_id"]
    )
    .reset_index(drop=True)
)

# Get the end time of each user's previous session
sessions["previous_session_end"] = (
    sessions.groupby("user_id")["session_end"]
    .shift(1)
)

# Calculate the time between consecutive sessions
sessions["inter_session_gap_seconds"] = (
    sessions["session_start"] -
    sessions["previous_session_end"]
).dt.total_seconds()

# Inspect the results
display(
    sessions[
        ["user_id", "session_id",
         "session_start", "session_end",
         "previous_session_end",
         "inter_session_gap_seconds"]
    ].head(15)
)

,user_id,session_id,session_start,session_end,previous_session_end,inter_session_gap_seconds
0,0,1,2018-01-16 06:01:05,2018-01-16 06:04:17,NaT,NaN
1,0,2,2018-01-16 06:25:54,2018-01-16 06:26:26,2018-01-16 06:04:17,1297.0
2,0,3,2018-01-16 06:35:35,2018-01-16 06:36:26,2018-01-16 06:26:26,549.0
3,0,4,2018-01-16 07:15:56,2018-01-16 07:21:44,2018-01-16 06:36:26,2370.0
4,0,5,2018-01-16 08:02:05,2018-01-16 08:04:11,2018-01-16 07:21:44,2421.0
5,0,6,2018-01-16 08:30:44,2018-01-16 08:34:14,2018-01-16 08:04:11,1593.0
6,0,7,2018-01-16 09:45:51,2018-01-16 09:51:45,2018-01-16 08:34:14,4297.0
7,0,8,2018-01-16 09:59:57,2018-01-16 09:59:57,2018-01-16 09:51:45,492.0
8,0,9,2018-01-16 10:29:54,2018-01-16 10:29:54,2018-01-16 09:59:57,1797.0
9,0,10,2018-01-16 11:01:28,2018-01-16 11:03:27,2018-01-16 10:29:54,1894.0


In [ ]:

# Count sessions with missing gaps
missing_gaps = (
    sessions["inter_session_gap_seconds"]
    .isna().sum()
)

# Examine zero, positive, and negative gaps
zero_gaps = (
    sessions["inter_session_gap_seconds"] == 0
).sum()

positive_gaps = (
    sessions["inter_session_gap_seconds"] > 0
).sum()

negative_gaps = (
    sessions["inter_session_gap_seconds"] < 0
).sum()

print("Missing gaps:", missing_gaps)
print("Zero gaps:", zero_gaps)
print("Positive gaps:", positive_gaps)
print("Negative gaps:", negative_gaps)

# Validate first-session missing values
assert missing_gaps == sessions["user_id"].nunique()

# Validate session preservation
assert len(sessions) == n_sessions

print("Temporal aggregation validated!")

# Examine gap distribution
display(
    sessions["inter_session_gap_seconds"].describe()
)

Missing gaps: 291
Zero gaps: 0
Positive gaps: 70688
Negative gaps: 0
Temporal aggregation validated!


,inter_session_gap_seconds
count,7.068800e+04
mean,5.247838e+03
std,1.009529e+05
min,3.010000e+02
25%,4.810000e+02
50%,8.200000e+02
75%,1.757000e+03
max,9.698402e+06


In [ ]:
# Calculate Rolling Session Frequency
# Count session starts within the previous 60 minutes
# Including the current session

def rolling_session_count(timestamps):
    return (
        pd.Series(1, index=pd.DatetimeIndex(timestamps))
        .rolling("60min", closed="both")
        .sum()
        .to_numpy()
    )

sessions["sessions_last_60min"] = (
    sessions.groupby("user_id")["session_start"]
    .transform(rolling_session_count)
    .astype(int)
)

# Inspect the new feature
display(
    sessions[
        ["user_id", "session_id", "session_start",
         "sessions_last_60min"]
    ].head(15)
)

,user_id,session_id,session_start,sessions_last_60min
0,0,1,2018-01-16 06:01:05,1
1,0,2,2018-01-16 06:25:54,2
2,0,3,2018-01-16 06:35:35,3
3,0,4,2018-01-16 07:15:56,3
4,0,5,2018-01-16 08:02:05,2
5,0,6,2018-01-16 08:30:44,2
6,0,7,2018-01-16 09:45:51,1
7,0,8,2018-01-16 09:59:57,2
8,0,9,2018-01-16 10:29:54,3
9,0,10,2018-01-16 11:01:28,2


In [ ]:
# Validate Rolling frequency
# Every session must count itself
assert (sessions["sessions_last_60min"] >= 1).all()

# Check that the number of rows remains unchanged
assert len(sessions) == n_sessions

print("Rolling session frequency validated!")

display(
    sessions["sessions_last_60min"].describe()
)

Rolling session frequency validated!


,sessions_last_60min
count,70979.000000
mean,2.987532
std,1.292485
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,9.000000


In [ ]:

# Count sessions starting within the previous 30 minutes
# Including the current session

def rolling_session_count_30min(timestamps):
    return (
        pd.Series(
            1,
            index=pd.DatetimeIndex(timestamps)
        )
        .rolling("30min", closed="both")
        .sum()
        .to_numpy()
    )

sessions["sessions_last_30min"] = (
    sessions.groupby("user_id")["session_start"]
    .transform(rolling_session_count_30min)
    .astype(int)
)

# Validate both rolling windows
assert len(sessions) == n_sessions

assert (sessions["sessions_last_30min"] >= 1).all()

assert (
    sessions["sessions_last_30min"]
    <= sessions["sessions_last_60min"]
).all()

print("30-minute session frequency validated!")

display(
    sessions[
        ["sessions_last_30min", "sessions_last_60min"]
    ].describe()
)

30-minute session frequency validated!


,sessions_last_30min,sessions_last_60min
count,70979.000000,70979.000000
mean,1.921427,2.987532
std,0.797768,1.292485
min,1.000000,1.000000
25%,1.000000,2.000000
50%,2.000000,3.000000
75%,2.000000,4.000000
max,5.000000,9.000000


In [ ]:

# Identify episodes lasting more than 0 but less than 15 seconds
df["is_short_episode"] = (
    (df["event_class"] == "episode") &
    (df["duration_seconds"] > 0) &
    (df["duration_seconds"] < 15)
)

# Check the total number of short episodes
print(
    "Total short episodes:",
    df["is_short_episode"].sum()
)

# Examine some examples
display(
    df.loc[
        df["is_short_episode"],
        ["user_id", "session_id", "app_name",
         "duration_seconds"]
    ].head(10)
)

Total short episodes: 346944


,user_id,session_id,app_name,duration_seconds
32,0,2,Google,5.0
33,0,2,Instagram,11.0
34,0,2,Google,5.0
48,0,4,Minesweeper Classic (Mines),1.0
135,0,6,Minesweeper Classic (Mines),1.0
217,0,10,Minesweeper Classic (Mines),1.0
288,0,15,Google Chrome,2.0
289,0,16,Google Chrome,11.0
292,0,19,Google Chrome,4.0
294,0,19,Minesweeper Classic (Mines),1.0


In [ ]:

# Aggregate short episodes by session
short_episodes = (
    df.groupby(["user_id", "session_id"], as_index=False)
    .agg(
        short_episode_count=("is_short_episode", "sum")
    )
)

# Merge into our existing session-level dataset
sessions = sessions.merge(
    short_episodes,
    on=["user_id", "session_id"],
    how="left",
    validate="one_to_one"
)

# Inspect the result
display(
    sessions[
        ["user_id", "session_id",
         "episode_count",
         "zero_duration_episode_count",
         "short_episode_count"]
    ].head(10)
)

,user_id,session_id,episode_count,zero_duration_episode_count,short_episode_count
0,0,1,31,31,0
1,0,2,3,0,3
2,0,3,1,0,0
3,0,4,36,35,1
4,0,5,39,39,0
5,0,6,41,40,1
6,0,7,40,40,0
7,0,8,0,0,0
8,0,9,0,0,0
9,0,10,32,31,1


In [ ]:

# Confirm all sessions were preserved
assert len(sessions) == n_sessions

# Short episodes cannot exceed positive-duration episodes
assert (
    sessions["short_episode_count"] <=
    (
        sessions["episode_count"] -
        sessions["zero_duration_episode_count"]
    )
).all()

# Validate total short episode count
assert (
    sessions["short_episode_count"].sum()
    == df["is_short_episode"].sum()
)

print("Short episode aggregation validated!")

# Examine the distribution
display(
    sessions["short_episode_count"].describe()
)


Short episode aggregation validated!


,short_episode_count
count,70979.000000
mean,4.887981
std,15.666222
min,0.000000
25%,0.000000
50%,1.000000
75%,4.000000
max,1014.000000


In [ ]:

# Identify the 10 sessions with the most short episodes
top_short_sessions = sessions.nlargest(
    10, "short_episode_count"
)

# Compare their behavioural measurements
display(
    top_short_sessions[
        ["user_id", "session_id",
         "episode_count",
         "zero_duration_episode_count",
         "short_episode_count",
         "recorded_episode_seconds",
         "session_span_seconds",
         "observed_app_transition_count"]
    ]
)


,user_id,session_id,episode_count,zero_duration_episode_count,short_episode_count,recorded_episode_seconds,session_span_seconds,observed_app_transition_count
47179,204,51034,1050,9,1014,3432.0,3622.0,20
1307,5,1559,946,12,883,5657.0,6014.0,43
1645,5,1928,940,14,830,7186.0,7496.0,26
65025,275,70059,935,7,805,7048.0,7261.0,28
1637,5,1918,971,31,754,11975.0,13029.0,95
48216,205,52164,1004,53,646,13531.0,15278.0,6
1669,5,1952,611,5,555,3611.0,3655.0,1
13211,68,14621,605,6,521,7672.0,8100.0,25
19313,112,21146,517,20,461,2711.0,3004.0,12
1294,5,1543,568,10,457,6256.0,6706.0,21


In [ ]:

# Select the session with the most short episodes
example = df[
    (df["user_id"] == 204) &
    (df["session_id"] == 51034) &
    (df["event_class"] == "episode")
].copy()

# Arrange episodes chronologically
example = example.sort_values(
    ["start_time", "session_event_number"]
)

# Identify the preceding application
example["previous_app"] = example["app_name"].shift(1)

# Check consecutive episodes involving the same app
example["same_app_as_previous"] = (
    example["app_name"] == example["previous_app"]
)

print("Total episodes:", len(example))

print(
    "Consecutive same-app episodes:",
    example["same_app_as_previous"].sum()
)

print("\nMost frequent applications:")
print(example["app_name"].value_counts().head(10))

print("\nEpisode duration statistics:")
print(example["duration_seconds"].describe())

# Inspect the first 20 episodes
print(
    example[
        ["app_name", "start_time", "end_time",
         "duration_seconds", "same_app_as_previous"]
    ].head(20).to_string(index=False)
)


Total episodes: 1050
Consecutive same-app episodes: 1033

Most frequent applications:
app_name
Settings            982
YouTube              44
Gmail                22
Verizon Messages      1
Google                1
Name: count, dtype: int64

Episode duration statistics:
count    1050.000000
mean        3.268571
std        11.143926
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max       279.000000
Name: duration_seconds, dtype: float64
app_name          start_time            end_time  duration_seconds  same_app_as_previous
 YouTube 2018-04-18 15:03:58 2018-04-18 15:04:03               5.0                 False
 YouTube 2018-04-18 15:04:03 2018-04-18 15:04:04               1.0                  True
 YouTube 2018-04-18 15:04:04 2018-04-18 15:04:09               5.0                  True
 YouTube 2018-04-18 15:04:09 2018-04-18 15:04:16               7.0                  True
 YouTube 2018-04-18 15:04:16 2018-04-18 15:05:12              56.0           

In [ ]:

# Ensure episodes are chronologically ordered
example = example.sort_values(
    ["start_time", "session_event_number"]
).copy()

# Retrieve the preceding episode's end time
example["previous_end_time"] = (
    example["end_time"].shift(1)
)

# Calculate the gap from the preceding episode
example["gap_from_previous_seconds"] = (
    example["start_time"] -
    example["previous_end_time"]
).dt.total_seconds()

# Identify consecutive episodes involving the same app
same_app = example["same_app_as_previous"]

print("Consecutive same-app pairs:", same_app.sum())

print(
    "Same-app pairs with zero gap:",
    (same_app &
     example["gap_from_previous_seconds"].eq(0)).sum()
)

print(
    "Same-app pairs with positive gap:",
    (same_app &
     example["gap_from_previous_seconds"].gt(0)).sum()
)

print(
    "Same-app pairs with overlapping intervals:",
    (same_app &
     example["gap_from_previous_seconds"].lt(0)).sum()
)

print("\nGap distribution for consecutive same-app pairs:")
print(
    example.loc[
        same_app, "gap_from_previous_seconds"
    ].describe()
)


Consecutive same-app pairs: 1033
Same-app pairs with zero gap: 979
Same-app pairs with positive gap: 54
Same-app pairs with overlapping intervals: 0

Gap distribution for consecutive same-app pairs:
count    1033.000000
mean        0.183930
std         3.352231
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       107.000000
Name: gap_from_previous_seconds, dtype: float64


In [ ]:

# Select all records from our investigated session
sample_events = df[
    (df["user_id"] == 204) &
    (df["session_id"] == 51034)
].copy()

# Extract episode records in chronological order
sample_episodes = (
    sample_events[
        sample_events["event_class"] == "episode"
    ]
    .sort_values(
        ["start_time", "session_event_number"],
        kind="stable"
    )
    .copy()
)

# Find previous episode details
sample_episodes["previous_app"] = (
    sample_episodes["app_name"].shift(1)
)

sample_episodes["previous_end"] = (
    sample_episodes["end_time"].shift(1)
)

# Identify contiguous same-app episode pairs
candidates = sample_episodes[
    sample_episodes["app_name"].eq(
        sample_episodes["previous_app"]
    )
    &
    sample_episodes["start_time"].eq(
        sample_episodes["previous_end"]
    )
].copy()

# Find applications observed at each event timestamp
apps_at_time = (
    sample_events.groupby("timestamp")["app_name"]
    .agg(lambda x: set(x.dropna()))
    .to_dict()
)

# Check whether another app was recorded
# at the shared episode boundary
candidates["other_app_at_boundary"] = [
    any(a != app for a in apps_at_time.get(t, set()))
    for t, app in zip(
        candidates["start_time"],
        candidates["app_name"]
    )
]

print("Contiguous same-app pairs:", len(candidates))

print(
    "Pairs with another app at boundary:",
    candidates["other_app_at_boundary"].sum()
)

print(
    "Pairs without another app at boundary:",
    (~candidates["other_app_at_boundary"]).sum()
)


Contiguous same-app pairs: 979
Pairs with another app at boundary: 1
Pairs without another app at boundary: 978


In [ ]:

# Use the episode ordering from Cell 34
work = sample_episodes.reset_index(drop=True).copy()

# Compare with the preceding episode
same_app = work["app_name"].eq(
    work["app_name"].shift(1)
)

touching = work["start_time"].eq(
    work["end_time"].shift(1)
)

# Check for another app at the shared boundary
work["conflicting_app"] = [
    any(a != app for a in apps_at_time.get(t, set()))
    for t, app in zip(
        work["start_time"],
        work["app_name"]
    )
]

# Conservative provisional merging condition
work["merge_with_previous"] = (
    same_app &
    touching &
    ~work["conflicting_app"]
)

# Begin a new bout whenever merging is not permitted
work["bout_id"] = (
    ~work["merge_with_previous"]
).cumsum()

# Aggregate consecutive fragments into bouts
conservative_bouts = (
    work.groupby("bout_id", as_index=False)
    .agg(
        app_name=("app_name", "first"),
        start_time=("start_time", "min"),
        end_time=("end_time", "max"),
        fragment_count=("app_name", "size")
    )
)

conservative_bouts["duration_seconds"] = (
    conservative_bouts["end_time"] -
    conservative_bouts["start_time"]
).dt.total_seconds()

print("Original episodes:", len(work))
print(
    "Accepted episode merges:",
    work["merge_with_previous"].sum()
)
print(
    "Conservative bouts:",
    len(conservative_bouts)
)
print(
    "Short bouts (1–14 seconds):",
    conservative_bouts["duration_seconds"]
    .between(1, 14).sum()
)

assert (
    len(conservative_bouts)
    == len(work) - work["merge_with_previous"].sum()
)

print("Bout count reconciliation passed!")


Original episodes: 1050
Accepted episode merges: 978
Conservative bouts: 72
Short bouts (1–14 seconds): 26
Bout count reconciliation passed!


In [ ]:

# Focus on bouts formed by merging multiple episodes
merged_bouts = conservative_bouts[
    conservative_bouts["fragment_count"] > 1
].copy()

def has_conflicting_activity(bout):

    # Events belonging to other applications
    other = sample_events[
        sample_events["app_name"] != bout["app_name"]
    ]

    # Interactions strictly inside this bout
    interactions_inside = (
        (other["event_class"] == "interaction") &
        (other["timestamp"] > bout["start_time"]) &
        (other["timestamp"] < bout["end_time"])
    ).any()

    # Other-app episode intervals overlapping this bout
    episodes_overlapping = (
        (other["event_class"] == "episode") &
        (other["start_time"] < bout["end_time"]) &
        (other["end_time"] > bout["start_time"])
    ).any()

    return interactions_inside or episodes_overlapping


merged_bouts["has_conflict"] = merged_bouts.apply(
    has_conflicting_activity,
    axis=1
)

print("Total provisional bouts:", len(conservative_bouts))
print("Bouts containing multiple fragments:", len(merged_bouts))
print(
    "Merged bouts with conflicting activity:",
    merged_bouts["has_conflict"].sum()
)

# Check that merging touching intervals preserves duration
print(
    "Total original episode seconds:",
    work["duration_seconds"].sum()
)

print(
    "Total reconstructed bout seconds:",
    conservative_bouts["duration_seconds"].sum()
)


Total provisional bouts: 72
Bouts containing multiple fragments: 47
Merged bouts with conflicting activity: 3
Total original episode seconds: 3432.0
Total reconstructed bout seconds: 3432.0


In [ ]:

# Select the three merged bouts flagged as conflicting
conflicted_bouts = merged_bouts[
    merged_bouts["has_conflict"]
].copy()

print("Conflicting bouts:")
print(
    conflicted_bouts[
        ["bout_id", "app_name", "start_time",
         "end_time", "fragment_count",
         "duration_seconds"]
    ].to_string(index=False)
)

# Inspect the other-app records responsible
for _, bout in conflicted_bouts.iterrows():

    other = sample_events[
        sample_events["app_name"] != bout["app_name"]
    ].copy()

    interaction_conflict = (
        (other["event_class"] == "interaction") &
        (other["timestamp"] > bout["start_time"]) &
        (other["timestamp"] < bout["end_time"])
    )

    episode_conflict = (
        (other["event_class"] == "episode") &
        (other["start_time"] < bout["end_time"]) &
        (other["end_time"] > bout["start_time"])
    )

    conflicts = other[
        interaction_conflict | episode_conflict
    ]

    print(
        f"\nBout {bout['bout_id']} "
        f"({bout['app_name']}):"
    )
    print("Conflicting records:", len(conflicts))

    print(
        conflicts[
            ["app_name", "event_class", "timestamp",
             "start_time", "end_time"]
        ].head(15).to_string(index=False)
    )


Conflicting bouts:
 bout_id app_name          start_time            end_time  fragment_count  duration_seconds
      35 Settings 2018-04-18 15:29:58 2018-04-18 15:30:41              18              43.0
      61 Settings 2018-04-18 15:46:20 2018-04-18 15:46:23               3               3.0
      66    Gmail 2018-04-18 15:46:28 2018-04-18 15:47:53              18              85.0

Bout 35 (Settings):
Conflicting records: 3
          app_name event_class           timestamp start_time end_time
Facebook Messenger interaction 2018-04-18 15:30:30        NaT      NaT
Facebook Messenger interaction 2018-04-18 15:30:30        NaT      NaT
Facebook Messenger interaction 2018-04-18 15:30:35        NaT      NaT

Bout 61 (Settings):
Conflicting records: 1
app_name event_class           timestamp          start_time            end_time
   Gmail     episode 2018-04-18 15:46:20 2018-04-18 15:46:20 2018-04-18 15:46:23

Bout 66 (Gmail):
Conflicting records: 1
app_name event_class           timesta

In [ ]:

# Audit every provisional bout, including single-fragment bouts
bout_audit = conservative_bouts.copy()

# Reuse the function defined in Cell 36
bout_audit["has_conflict"] = bout_audit.apply(
    has_conflicting_activity,
    axis=1
)

# Identify positive-duration short bouts
bout_audit["is_short_bout"] = (
    (bout_audit["duration_seconds"] > 0) &
    (bout_audit["duration_seconds"] < 15)
)

print("Total provisional bouts:", len(bout_audit))

print(
    "Bouts with conflicting activity:",
    bout_audit["has_conflict"].sum()
)

print(
    "All short bouts:",
    bout_audit["is_short_bout"].sum()
)

print(
    "Short bouts with conflicts:",
    (
        bout_audit["is_short_bout"] &
        bout_audit["has_conflict"]
    ).sum()
)

print(
    "Short bouts without detected conflicts:",
    (
        bout_audit["is_short_bout"] &
        ~bout_audit["has_conflict"]
    ).sum()
)


Total provisional bouts: 72
Bouts with conflicting activity: 10
All short bouts: 26
Short bouts with conflicts: 8
Short bouts without detected conflicts: 18


In [ ]:

session_keys = ["user_id", "session_id"]

# Count distinct apps observed at each event timestamp
apps_per_timestamp = (
    df.groupby(session_keys + ["timestamp"])
    ["app_name"]
    .nunique()
    .rename("apps_at_timestamp")
    .reset_index()
)

# Select episode records without modifying df
all_episodes = df.loc[
    df["event_class"] == "episode",
    session_keys + [
        "session_event_number",
        "app_name",
        "start_time",
        "end_time",
        "duration_seconds"
    ]
].copy()

# Attach timestamp-level app diversity
all_episodes = all_episodes.merge(
    apps_per_timestamp,
    left_on=session_keys + ["start_time"],
    right_on=session_keys + ["timestamp"],
    how="left",
    validate="many_to_one"
).drop(columns="timestamp")

# Consistent chronological ordering
all_episodes = all_episodes.sort_values(
    session_keys + ["start_time", "session_event_number"],
    kind="stable"
).reset_index(drop=True)

grouped = all_episodes.groupby(session_keys)

# Compare each episode with its predecessor
previous_app = grouped["app_name"].shift(1)
previous_end = grouped["end_time"].shift(1)

same_app = all_episodes["app_name"].eq(previous_app)

touching = all_episodes["start_time"].eq(previous_end)

no_boundary_conflict = (
    all_episodes["apps_at_timestamp"].eq(1)
)

# Apply our provisional merging rule
all_episodes["merge_with_previous"] = (
    same_app &
    touching &
    no_boundary_conflict
)

all_episodes["new_bout"] = (
    ~all_episodes["merge_with_previous"]
)

# Number bouts independently within each session
all_episodes["bout_id"] = (
    all_episodes.groupby(session_keys)["new_bout"]
    .cumsum()
)

# Aggregate episode fragments into bouts
all_bouts = (
    all_episodes.groupby(
        session_keys + ["bout_id"],
        as_index=False
    )
    .agg(
        app_name=("app_name", "first"),
        start_time=("start_time", "min"),
        end_time=("end_time", "max"),
        fragment_count=("app_name", "size")
    )
)

all_bouts["duration_seconds"] = (
    all_bouts["end_time"] -
    all_bouts["start_time"]
).dt.total_seconds()

# Basic validation
assert all_bouts["fragment_count"].sum() == len(all_episodes)

assert (
    len(all_bouts)
    == len(all_episodes)
    - all_episodes["merge_with_previous"].sum()
)

print("Original episodes:", len(all_episodes))
print("Accepted merges:", all_episodes["merge_with_previous"].sum())
print("Provisional bouts:", len(all_bouts))

print(
    "Short episodes:",
    all_episodes["duration_seconds"]
    .between(1, 14).sum()
)

print(
    "Short bouts:",
    all_bouts["duration_seconds"]
    .between(1, 14).sum()
)

print(
    "Unmatched boundary timestamps:",
    all_episodes["apps_at_timestamp"].isna().sum()
)

print(
    "Original total duration:",
    all_episodes["duration_seconds"].sum()
)

print(
    "Bout total duration:",
    all_bouts["duration_seconds"].sum()
)


Original episodes: 585520
Accepted merges: 250907
Provisional bouts: 334613
Short episodes: 346944
Short bouts: 131808
Unmatched boundary timestamps: 0
Original total duration: 14161098.0
Bout total duration: 14161098.0


In [ ]:

# Identify short bouts
all_bouts["is_short_bout"] = (
    (all_bouts["duration_seconds"] > 0) &
    (all_bouts["duration_seconds"] < 15)
)

# Aggregate bouts per session
bout_features = (
    all_bouts.groupby(
        ["user_id", "session_id"],
        as_index=False
    )
    .agg(
        bout_count=("bout_id", "size"),
        short_bout_count=("is_short_bout", "sum")
    )
)

# Compare original and reconstructed measurements
comparison = sessions[
    ["user_id", "session_id",
     "episode_count", "short_episode_count"]
].merge(
    bout_features,
    on=["user_id", "session_id"],
    how="left",
    validate="one_to_one"
)

# Sessions without episodes also have no bouts
comparison[["bout_count", "short_bout_count"]] = (
    comparison[["bout_count", "short_bout_count"]]
    .fillna(0)
    .astype(int)
)

# Calculate reductions
comparison["episode_reduction"] = (
    comparison["episode_count"] -
    comparison["bout_count"]
)

comparison["short_reduction"] = (
    comparison["short_episode_count"] -
    comparison["short_bout_count"]
)

# Validate
assert len(comparison) == len(sessions)
assert (comparison["episode_reduction"] >= 0).all()

assert comparison["bout_count"].sum() == len(all_bouts)

print("Session-level comparison validated!")

print(
    "\nSessions containing episodes:",
    (comparison["episode_count"] > 0).sum()
)

print(
    "Sessions with reduced episode counts:",
    (comparison["episode_reduction"] > 0).sum()
)

print(
    "Sessions with reduced short-use counts:",
    (comparison["short_reduction"] > 0).sum()
)

print("\nDistribution of counts:")
print(
    comparison[
        ["episode_count", "bout_count",
         "short_episode_count", "short_bout_count"]
    ].describe(percentiles=[.5, .75, .9, .95, .99]).round(2)
    .to_string()
)


Session-level comparison validated!

Sessions containing episodes: 58697
Sessions with reduced episode counts: 28132
Sessions with reduced short-use counts: 27047

Distribution of counts:
       episode_count  bout_count  short_episode_count  short_bout_count
count       70979.00    70979.00             70979.00          70979.00
mean            8.25        4.71                 4.89              1.86
std            20.79        8.33                15.67              3.90
min             0.00        0.00                 0.00              0.00
50%             3.00        2.00                 1.00              1.00
75%             8.00        5.00                 4.00              2.00
90%            19.00       11.00                11.00              5.00
95%            32.00       17.00                19.00              7.00
99%            81.00       39.00                54.00             17.00
max          1050.00      245.00              1014.00            186.00


The initial episode reconstruction exhibited sensitivity to consecutive same-application boundaries. A continuity-aware bout representation was therefore introduced as an alternative analytical unit. Across the LSApp dataset, this reduced total application-use units by approximately 42.9% and short-use units by approximately 62.0%, while preserving aggregate recorded duration. These bout measures remain provisional pending validation of conflicting application activity.

In [ ]:

import numpy as np

# Create a separate audit table
bout_audit = all_bouts.reset_index(drop=True).copy()

bout_audit["cross_app_overlap"] = False

# Compare bouts only within the same session
for _, group in bout_audit.groupby(
    ["user_id", "session_id"], sort=False
):

    # A conflict requires at least two applications
    if len(group) < 2 or group["app_name"].nunique() < 2:
        continue

    starts = group["start_time"].to_numpy()
    ends = group["end_time"].to_numpy()
    apps = group["app_name"].to_numpy()

    # Only positive-duration intervals are compared
    positive = ends > starts

    # Pairwise interval overlap
    overlaps = (
        (starts[:, None] < ends[None, :]) &
        (ends[:, None] > starts[None, :]) &
        (apps[:, None] != apps[None, :]) &
        positive[:, None] &
        positive[None, :]
    )

    bout_audit.loc[
        group.index, "cross_app_overlap"
    ] = overlaps.any(axis=1)

# Identify short bouts
bout_audit["is_short_bout"] = (
    bout_audit["duration_seconds"].between(1, 14)
)

print("Total bouts:", len(bout_audit))

print(
    "Bouts with cross-app overlap:",
    bout_audit["cross_app_overlap"].sum()
)

print(
    "Sessions containing overlapping apps:",
    bout_audit.loc[
        bout_audit["cross_app_overlap"],
        ["user_id", "session_id"]
    ].drop_duplicates().shape[0]
)

print(
    "Short bouts with cross-app overlap:",
    (
        bout_audit["is_short_bout"] &
        bout_audit["cross_app_overlap"]
    ).sum()
)

print("Interval overlap audit completed!")


Total bouts: 334613
Bouts with cross-app overlap: 13497
Sessions containing overlapping apps: 2758
Short bouts with cross-app overlap: 8688
Interval overlap audit completed!


In [ ]:

# Prepare interaction timestamps by session and application
interaction_lookup = {}

interactions = df[
    df["event_class"] == "interaction"
].dropna(subset=["timestamp", "app_name"])

for key, group in interactions.groupby(
    ["user_id", "session_id"], sort=False
):
    all_times = np.sort(
        group["timestamp"].to_numpy(dtype="datetime64[ns]")
    )

    by_app = {
        app: np.sort(
            rows["timestamp"].to_numpy(dtype="datetime64[ns]")
        )
        for app, rows in group.groupby("app_name")
    }

    interaction_lookup[key] = (all_times, by_app)


# Audit every reconstructed bout
bout_audit["other_app_interaction_inside"] = False

for key, group in bout_audit.groupby(
    ["user_id", "session_id"], sort=False
):
    if key not in interaction_lookup:
        continue

    all_times, by_app = interaction_lookup[key]

    starts = group["start_time"].to_numpy(
        dtype="datetime64[ns]"
    )
    ends = group["end_time"].to_numpy(
        dtype="datetime64[ns]"
    )

    # Count all interactions strictly inside each bout
    total_inside = (
        np.searchsorted(all_times, ends, side="left")
        - np.searchsorted(all_times, starts, side="right")
    )

    # Subtract interactions belonging to the bout's own app
    same_app_inside = np.zeros(len(group), dtype=int)

    for app in group["app_name"].dropna().unique():
        app_times = by_app.get(app)

        if app_times is None:
            continue

        mask = group["app_name"].eq(app).to_numpy()

        same_app_inside[mask] = (
            np.searchsorted(
                app_times, ends[mask], side="left"
            )
            - np.searchsorted(
                app_times, starts[mask], side="right"
            )
        )

    bout_audit.loc[
        group.index, "other_app_interaction_inside"
    ] = total_inside > same_app_inside


# Combine the two conflict indicators
bout_audit["any_detected_conflict"] = (
    bout_audit["cross_app_overlap"] |
    bout_audit["other_app_interaction_inside"]
)

print(
    "Bouts with other-app interactions inside:",
    bout_audit["other_app_interaction_inside"].sum()
)

print(
    "Bouts with either detected conflict:",
    bout_audit["any_detected_conflict"].sum()
)

print(
    "Short bouts with either detected conflict:",
    (
        bout_audit["is_short_bout"] &
        bout_audit["any_detected_conflict"]
    ).sum()
)

print(
    "Short bouts without either detected conflict:",
    (
        bout_audit["is_short_bout"] &
        ~bout_audit["any_detected_conflict"]
    ).sum()
)


Bouts with other-app interactions inside: 20161
Bouts with either detected conflict: 32305
Short bouts with either detected conflict: 12959
Short bouts without either detected conflict: 118849


In [ ]:
sessions.columns

Index(['user_id', 'session_id', 'first_event_time', 'last_event_time',
       'event_count', 'episode_count', 'interaction_count',
       'zero_duration_episode_count', 'event_span_seconds',
       'recorded_episode_seconds', 'session_start', 'session_end',
       'session_span_seconds', 'unique_apps_observed', 'unique_episode_apps',
       'unique_interaction_apps', 'observed_app_transition_count',
       'observed_app_revisit_count', 'previous_session_end',
       'inter_session_gap_seconds', 'sessions_last_60min',
       'sessions_last_30min', 'short_episode_count'],
      dtype='object')

In [ ]:

# Prepare additional bout-level indicators
bout_audit["is_short_bout_unflagged"] = (
    bout_audit["is_short_bout"] &
    ~bout_audit["any_detected_conflict"]
)

bout_audit["is_short_bout_conflicted"] = (
    bout_audit["is_short_bout"] &
    bout_audit["any_detected_conflict"]
)

# Aggregate reconstructed bouts per session
bout_session_features = (
    bout_audit.groupby(
        ["user_id", "session_id"],
        as_index=False
    )
    .agg(
        bout_count=("bout_id", "size"),
        short_bout_count=("is_short_bout", "sum"),
        conflicted_bout_count=("any_detected_conflict", "sum"),
        short_bout_unflagged_count=("is_short_bout_unflagged", "sum"),
        short_bout_conflicted_count=("is_short_bout_conflicted", "sum")
    )
)

# Merge features into the existing session table
sessions = sessions.merge(
    bout_session_features,
    on=["user_id", "session_id"],
    how="left",
    validate="one_to_one"
)

# Sessions without episodes have no reconstructed bouts
bout_columns = [
    "bout_count",
    "short_bout_count",
    "conflicted_bout_count",
    "short_bout_unflagged_count",
    "short_bout_conflicted_count"
]

sessions[bout_columns] = (
    sessions[bout_columns].fillna(0).astype(int)
)

# Validate the integration
assert len(sessions) == n_sessions

assert sessions["bout_count"].sum() == len(bout_audit)

assert (
    sessions["short_bout_count"].sum() == 131808
)

assert (
    sessions["short_bout_unflagged_count"].sum() == 118849
)

assert (
    sessions["short_bout_count"]
    == sessions["short_bout_unflagged_count"]
    + sessions["short_bout_conflicted_count"]
).all()

assert (
    sessions["bout_count"] <= sessions["episode_count"]
).all()

print("Bout-based features successfully integrated!")

print("\nFeature distributions:")
print(
    sessions[bout_columns]
    .describe()
    .round(2)
    .to_string()
)


Bout-based features successfully integrated!

Feature distributions:
       bout_count  short_bout_count  conflicted_bout_count  short_bout_unflagged_count  short_bout_conflicted_count
count    70979.00          70979.00               70979.00                    70979.00                     70979.00
mean         4.71              1.86                   0.46                        1.67                         0.18
std          8.33              3.90                   2.07                        3.21                         1.25
min          0.00              0.00                   0.00                        0.00                         0.00
25%          1.00              0.00                   0.00                        0.00                         0.00
50%          2.00              1.00                   0.00                        1.00                         0.00
75%          5.00              2.00                   0.00                        2.00                         0.00
max

In [ ]:
# Select short bouts without detected conflicts
eligible_bouts = bout_audit[
    bout_audit["is_short_bout"] &
    ~bout_audit["any_detected_conflict"]
].copy()

# Store sorted bout completion times for each user
bout_lookup = {
    user_id: np.sort(
        group["end_time"].to_numpy(
            dtype="datetime64[ns]"
        )
    )
    for user_id, group in eligible_bouts.groupby("user_id")
}

# Initialize rolling frequency features
sessions["prior_short_bouts_30min"] = 0
sessions["prior_short_bouts_60min"] = 0

# Calculate frequencies at each session's START
for user_id, indices in sessions.groupby("user_id").groups.items():

    session_starts = sessions.loc[
        indices, "session_start"
    ].to_numpy(dtype="datetime64[ns]")

    bout_ends = bout_lookup.get(
        user_id,
        np.array([], dtype="datetime64[ns]")
    )

    # Number of completed bouts strictly before session start
    right = np.searchsorted(
        bout_ends, session_starts, side="left"
    )

    # Earliest included bout in each window
    left_30 = np.searchsorted(
        bout_ends,
        session_starts - np.timedelta64(30, "m"),
        side="left"
    )

    left_60 = np.searchsorted(
        bout_ends,
        session_starts - np.timedelta64(60, "m"),
        side="left"
    )

    sessions.loc[
        indices, "prior_short_bouts_30min"
    ] = right - left_30

    sessions.loc[
        indices, "prior_short_bouts_60min"
    ] = right - left_60


# Validate
assert len(sessions) == n_sessions

assert (
    sessions["prior_short_bouts_30min"] >= 0
).all()

assert (
    sessions["prior_short_bouts_30min"]
    <= sessions["prior_short_bouts_60min"]
).all()

# The first recorded session of each user has no earlier bouts
first_sessions = sessions.sort_values(
    ["user_id", "session_start", "session_id"]
).groupby("user_id").head(1)

assert (
    first_sessions["prior_short_bouts_60min"] == 0
).all()

print("Rolling short-bout frequencies validated!")

print("\nDistributions:")
print(
    sessions[
        ["prior_short_bouts_30min",
         "prior_short_bouts_60min"]
    ].describe(percentiles=[.5, .75, .9, .95, .99])
    .round(2)
    .to_string()
)


Rolling short-bout frequencies validated!

Distributions:
       prior_short_bouts_30min  prior_short_bouts_60min
count                 70979.00                 70979.00
mean                      1.54                     3.37
std                       2.62                     4.73
min                       0.00                     0.00
50%                       0.00                     2.00
75%                       2.00                     5.00
90%                       4.00                     9.00
95%                       6.00                    12.00
99%                      12.00                    21.00
max                      74.00                    97.00


In [ ]:

# Exclude the current session from session frequency
sessions["prior_sessions_60min"] = (
    sessions["sessions_last_60min"] - 1
)

# Validate
assert (sessions["prior_sessions_60min"] >= 0).all()

# Compare the two frequency measurements
correlation = sessions[
    ["prior_sessions_60min", "prior_short_bouts_60min"]
].corr(method="spearman")

print("Spearman correlation:")
print(correlation.round(3).to_string())

# Examine short-bout frequency at each session-frequency level
comparison_summary = (
    sessions.groupby("prior_sessions_60min")
    .agg(
        session_count=("session_id", "size"),
        median_short_bouts=("prior_short_bouts_60min", "median"),
        mean_short_bouts=("prior_short_bouts_60min", "mean"),
        p90_short_bouts=(
            "prior_short_bouts_60min",
            lambda x: x.quantile(0.90)
        ),
        max_short_bouts=("prior_short_bouts_60min", "max")
    )
)

print("\nFrequency comparison:")
print(comparison_summary.round(2).to_string())


Spearman correlation:
                         prior_sessions_60min  prior_short_bouts_60min
prior_sessions_60min                     1.00                     0.45
prior_short_bouts_60min                  0.45                     1.00

Frequency comparison:
                      session_count  median_short_bouts  mean_short_bouts  p90_short_bouts  max_short_bouts
prior_sessions_60min                                                                                       
0                             10089                 0.0              0.59              0.0               85
1                             16221                 1.0              2.69              7.0               97
2                             20068                 2.0              3.91              9.0               86
3                             15769                 3.0              4.41             10.0               90
4                              6981                 4.0              4.64             10.0    

In [ ]:

# Exclude the current session from the 30-minute count
sessions["prior_sessions_30min"] = (
    sessions["sessions_last_30min"] - 1
)

# Select features available at session start
decision_columns = [
    "user_id",
    "session_id",
    "session_start",
    "inter_session_gap_seconds",
    "prior_sessions_30min",
    "prior_sessions_60min",
    "prior_short_bouts_30min",
    "prior_short_bouts_60min"
]

decision_features = sessions[decision_columns].copy()

# Validate
assert len(decision_features) == n_sessions

assert (decision_features["prior_sessions_30min"] >= 0).all()

assert (
    decision_features["prior_sessions_30min"]
    <= decision_features["prior_sessions_60min"]
).all()

# Only the first observed session per user should lack
# a previous-session gap
assert (
    decision_features["inter_session_gap_seconds"]
    .isna().sum() == sessions["user_id"].nunique()
)

print("Decision-point features validated!")

print("\nMissing values:")
print(decision_features.isna().sum().to_string())

print("\nFirst 10 observations:")
print(decision_features.head(10).to_string(index=False))


Decision-point features validated!

Missing values:
user_id                        0
session_id                     0
session_start                  0
inter_session_gap_seconds    291
prior_sessions_30min           0
prior_sessions_60min           0
prior_short_bouts_30min        0
prior_short_bouts_60min        0

First 10 observations:
 user_id  session_id       session_start  inter_session_gap_seconds  prior_sessions_30min  prior_sessions_60min  prior_short_bouts_30min  prior_short_bouts_60min
       0           1 2018-01-16 06:01:05                        NaN                     0                     0                        0                        0
       0           2 2018-01-16 06:25:54                     1297.0                     1                     1                        0                        0
       0           3 2018-01-16 06:35:35                      549.0                     1                     2                        3                        3
       0    

In [ ]:

# Extract temporal context from session start
decision_features["hour_of_day"] = (
    decision_features["session_start"].dt.hour
)

decision_features["day_of_week"] = (
    decision_features["session_start"].dt.dayofweek
)

# Mark sessions with no recorded predecessor
decision_features["first_observed_session"] = (
    decision_features["inter_session_gap_seconds"].isna()
)

# Validation
assert decision_features["hour_of_day"].between(0, 23).all()

assert decision_features["day_of_week"].between(0, 6).all()

assert (
    decision_features["first_observed_session"].sum()
    == decision_features["user_id"].nunique()
)

print("Temporal context features validated!")

print("\nFirst observed sessions:",
      decision_features["first_observed_session"].sum())

print("\nSession starts by hour:")
print(
    decision_features["hour_of_day"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSession starts by day of week:")
print(
    decision_features["day_of_week"]
    .value_counts()
    .sort_index()
    .to_string()
)


Temporal context features validated!

First observed sessions: 291

Session starts by hour:
hour_of_day
0     3561
1     3275
2     3065
3     2681
4     2313
5     2047
6     1633
7     1526
8     1408
9     1485
10    1910
11    2414
12    2813
13    3280
14    3424
15    3653
16    3849
17    3832
18    3965
19    3920
20    3906
21    3841
22    3671
23    3507

Session starts by day of week:
day_of_week
0    10701
1    10551
2    10326
3     9882
4     9638
5     9996
6     9885


### Feature extraction: history baselines at 75th percentile

In [ ]:

# Ensure chronological order within each user
decision_features = decision_features.sort_values(
    ["user_id", "session_start", "session_id"]
).reset_index(drop=True)

# Count how many earlier session observations exist
decision_features["history_session_count"] = (
    decision_features.groupby("user_id").cumcount()
)

# Historical 75th percentile for session frequency
decision_features["baseline_sessions_p75"] = (
    decision_features.groupby("user_id")[
        "prior_sessions_60min"
    ].transform(
        lambda s: s.shift(1)
        .expanding(min_periods=10)
        .quantile(0.75)
    )
)

# Historical 75th percentile for short-bout frequency
decision_features["baseline_short_bouts_p75"] = (
    decision_features.groupby("user_id")[
        "prior_short_bouts_60min"
    ].transform(
        lambda s: s.shift(1)
        .expanding(min_periods=10)
        .quantile(0.75)
    )
)

# Validate
assert len(decision_features) == n_sessions

baseline_columns = [
    "baseline_sessions_p75",
    "baseline_short_bouts_p75"
]

assert (
    decision_features.loc[
        decision_features["history_session_count"] < 10,
        baseline_columns
    ].isna().all().all()
)

assert (
    decision_features.loc[
        decision_features["history_session_count"] >= 10,
        baseline_columns
    ].notna().all().all()
)

print("Historical baselines validated!")

print(
    "\nObservations with available baselines:",
    decision_features["baseline_short_bouts_p75"]
    .notna().sum()
)

print("\nBaseline distributions:")
print(
    decision_features[baseline_columns]
    .describe()
    .round(2)
    .to_string()
)


Historical baselines validated!

Observations with available baselines: 68147

Baseline distributions:
       baseline_sessions_p75  baseline_short_bouts_p75
count               68147.00                  68147.00
mean                    2.75                      4.87
std                     0.54                      2.91
min                     0.00                      0.00
25%                     2.25                      3.00
50%                     3.00                      4.00
75%                     3.00                      6.00
max                     7.25                     39.50


In [ ]:

# Identify observations with sufficient history
has_baseline = (
    decision_features["baseline_sessions_p75"].notna() &
    decision_features["baseline_short_bouts_p75"].notna()
)

# Flag session frequency above personal baseline
decision_features["elevated_session_frequency"] = (
    (
        decision_features["prior_sessions_60min"] >
        decision_features["baseline_sessions_p75"]
    )
    .astype("boolean")
    .where(has_baseline)
)

# Flag short-bout frequency above personal baseline
decision_features["elevated_short_bout_frequency"] = (
    (
        decision_features["prior_short_bouts_60min"] >
        decision_features["baseline_short_bouts_p75"]
    )
    .astype("boolean")
    .where(has_baseline)
)

# Validate missing-history handling
assert (
    decision_features.loc[
        ~has_baseline,
        ["elevated_session_frequency",
         "elevated_short_bout_frequency"]
    ].isna().all().all()
)

assert (
    decision_features.loc[
        has_baseline,
        ["elevated_session_frequency",
         "elevated_short_bout_frequency"]
    ].notna().all().all()
)

print("Personalised engagement indicators validated!")

print("\nObservations with sufficient history:",
      has_baseline.sum())

print("\nElevated engagement rates:")
print(
    decision_features.loc[
        has_baseline,
        ["elevated_session_frequency",
         "elevated_short_bout_frequency"]
    ].mean().round(3).to_string()
)

print("\nJoint frequency patterns:")
print(
    pd.crosstab(
        decision_features.loc[
            has_baseline, "elevated_session_frequency"
        ],
        decision_features.loc[
            has_baseline, "elevated_short_bout_frequency"
        ],
        margins=True
    ).to_string()
)


Personalised engagement indicators validated!

Observations with sufficient history: 68147

Elevated engagement rates:
elevated_session_frequency        0.17
elevated_short_bout_frequency    0.225

Joint frequency patterns:
elevated_short_bout_frequency  False   True    All
elevated_session_frequency                        
False                          45345  11226  56571
True                            7466   4110  11576
All                            52811  15336  68147


### The Feature Extraction: Historical Baselines at 90th percentile

In [ ]:

# Calculate historical 90th-percentile baselines
# using only observations preceding the current one

decision_features["baseline_sessions_p90"] = (
    decision_features.groupby("user_id")[
        "prior_sessions_60min"
    ].transform(
        lambda s: s.shift(1)
        .expanding(min_periods=10)
        .quantile(0.90)
    )
)

decision_features["baseline_short_bouts_p90"] = (
    decision_features.groupby("user_id")[
        "prior_short_bouts_60min"
    ].transform(
        lambda s: s.shift(1)
        .expanding(min_periods=10)
        .quantile(0.90)
    )
)

# Use the same observations as our 75th-percentile analysis
eligible = decision_features["history_session_count"] >= 10

# Define elevated engagement using the stricter threshold
decision_features["elevated_session_frequency_p90"] = (
    decision_features["prior_sessions_60min"] >
    decision_features["baseline_sessions_p90"]
).astype("boolean").where(eligible)

decision_features["elevated_short_bout_frequency_p90"] = (
    decision_features["prior_short_bouts_60min"] >
    decision_features["baseline_short_bouts_p90"]
).astype("boolean").where(eligible)

# Validate
assert (
    decision_features.loc[
        eligible,
        ["elevated_session_frequency_p90",
         "elevated_short_bout_frequency_p90"]
    ].notna().all().all()
)

# Compare the 75th and 90th percentile definitions
comparison = decision_features.loc[eligible]

for threshold, session_col, bout_col in [
    ("P75",
     "elevated_session_frequency",
     "elevated_short_bout_frequency"),
    ("P90",
     "elevated_session_frequency_p90",
     "elevated_short_bout_frequency_p90")
]:
    print(f"\n{threshold} threshold:")

    print("Elevated session frequency:",
          round(comparison[session_col].mean() * 100, 2), "%")

    print("Elevated short-bout frequency:",
          round(comparison[bout_col].mean() * 100, 2), "%")

    print("\nJoint patterns:")
    print(pd.crosstab(
        comparison[session_col],
        comparison[bout_col]
    ).to_string())



P75 threshold:
Elevated session frequency: 16.99 %
Elevated short-bout frequency: 22.5 %

Joint patterns:
elevated_short_bout_frequency  False  True 
elevated_session_frequency                 
False                          45345  11226
True                            7466   4110

P90 threshold:
Elevated session frequency: 6.45 %
Elevated short-bout frequency: 9.93 %

Joint patterns:
elevated_short_bout_frequency_p90  False  True 
elevated_session_frequency_p90                 
False                              57776   5976
True                                3606    789


### Assess Baseline Sensitivity

In [ ]:

# Compare personal baselines after 10 and 50
# previously observed session starts

baseline_cols = [
    "baseline_sessions_p75",
    "baseline_short_bouts_p75",
    "baseline_sessions_p90",
    "baseline_short_bouts_p90"
]

snapshots = decision_features[
    decision_features["history_session_count"].isin([10, 50])
][
    ["user_id", "history_session_count"] + baseline_cols
].copy()

# Reshape to compare both historical stages
baseline_comparison = snapshots.pivot(
    index="user_id",
    columns="history_session_count",
    values=baseline_cols
).dropna()

print(
    "Users with both baseline snapshots:",
    len(baseline_comparison)
)

print("\nAbsolute changes from 10 to 50 observations:")

for col in baseline_cols:
    changes = (
        baseline_comparison[(col, 50)] -
        baseline_comparison[(col, 10)]
    ).abs()

    print(f"\n{col}")
    print("Median absolute change:", round(changes.median(), 2))
    print("90th percentile change:", round(changes.quantile(.90), 2))
    print("Maximum change:", round(changes.max(), 2))


Users with both baseline snapshots: 196

Absolute changes from 10 to 50 observations:

baseline_sessions_p75
Median absolute change: 0.75
90th percentile change: 1.25
Maximum change: 3.0

baseline_short_bouts_p75
Median absolute change: 1.38
90th percentile change: 5.62
Maximum change: 20.0

baseline_sessions_p90
Median absolute change: 0.9
90th percentile change: 1.9
Maximum change: 3.0

baseline_short_bouts_p90
Median absolute change: 2.65
90th percentile change: 7.65
Maximum change: 17.2


### Evaluate an intermediate warm-up period

In [ ]:

# Part A: Assess participant coverage
print("Warm-up coverage:")

for n in [10, 20, 30, 50]:
    eligible = decision_features[
        "history_session_count"
    ] >= n

    print(
        f"{n} observations: "
        f"{eligible.sum()} session starts, "
        f"{decision_features.loc[eligible, 'user_id'].nunique()} users"
    )


# Part B: Compare intermediate baselines to 50
baseline_cols = [
    "baseline_short_bouts_p75",
    "baseline_short_bouts_p90"
]

snapshots = (
    decision_features.loc[
        decision_features["history_session_count"]
        .isin([20, 30, 50]),
        ["user_id", "history_session_count"] + baseline_cols
    ]
    .pivot(
        index="user_id",
        columns="history_session_count",
        values=baseline_cols
    )
    .dropna()
)

print("\nUsers with all three snapshots:", len(snapshots))

for col in baseline_cols:
    print(f"\n{col}")

    for n in [20, 30]:
        changes = (
            snapshots[(col, 50)] -
            snapshots[(col, n)]
        ).abs()

        print(
            f"{n} to 50 observations: "
            f"median change = {changes.median():.2f}, "
            f"P90 change = {changes.quantile(.90):.2f}"
        )


Warm-up coverage:
10 observations: 68147 session starts, 271 users
20 observations: 65513 session starts, 255 users
30 observations: 63054 session starts, 240 users
50 observations: 58615 session starts, 196 users

Users with all three snapshots: 196

baseline_short_bouts_p75
20 to 50 observations: median change = 1.00, P90 change = 4.25
30 to 50 observations: median change = 1.00, P90 change = 3.25

baseline_short_bouts_p90
20 to 50 observations: median change = 1.80, P90 change = 6.10
30 to 50 observations: median change = 1.00, P90 change = 4.95


In [ ]:
# Require at least 30 previous observations
eligible_30 = (
    decision_features["history_session_count"] >= 30
)

# Compare recent behaviour against historical P75 baselines
session_elevated = (
    decision_features["prior_sessions_60min"] >
    decision_features["baseline_sessions_p75"]
)

bout_elevated = (
    decision_features["prior_short_bouts_60min"] >
    decision_features["baseline_short_bouts_p75"]
)

# Construct mutually exclusive engagement patterns
conditions = [
    ~eligible_30,
    session_elevated & bout_elevated,
    session_elevated & ~bout_elevated,
    ~session_elevated & bout_elevated
]

labels = [
    "insufficient_history",
    "both_elevated",
    "session_only",
    "short_bout_only"
]

decision_features["engagement_pattern_p75"] = np.select(
    conditions,
    labels,
    default="neither_elevated"
)

# Validate classification
assert len(decision_features) == n_sessions

assert (
    decision_features["engagement_pattern_p75"]
    .eq("insufficient_history").sum()
    == (~eligible_30).sum()
)

assert (
    decision_features.loc[
        eligible_30, "engagement_pattern_p75"
    ].ne("insufficient_history").all()
)

print("Engagement patterns validated!")

print("\nPattern distribution:")
print(
    decision_features["engagement_pattern_p75"]
    .value_counts()
    .to_string()
)

print("\nPercentages:")
print(
    (
        decision_features["engagement_pattern_p75"]
        .value_counts(normalize=True) * 100
    ).round(2).to_string()
)


Engagement patterns validated!

Pattern distribution:
engagement_pattern_p75
neither_elevated        42372
short_bout_only         10324
insufficient_history     7925
session_only             6849
both_elevated            3509

Percentages:
engagement_pattern_p75
neither_elevated        59.70
short_bout_only         14.55
insufficient_history    11.17
session_only             9.65
both_elevated            4.94
